In [1]:
# Compartison of table from piano and from orchestra
# FM 07/10/2026

import sys
sys.path.append('amos') 
# Import all ML orchestration functions
from amos.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amos.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amos.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amos.ml_orchestration import amo

/home/francesco/anaconda3/envs/auto-orch/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
2025-10-08 09:31:08.477159: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
#filepiano='midis/5beet1mv_solo.mid'
#fileorch='midis/symphony_5_1_orch.mid'

filepiano='midis/liszt_classical_archives-0/piano.mid'
fileorch='midis/liszt_classical_archives-0/orchestra.mid'

In [3]:
print(f"Orchestra file: {fileorch}")
dforch = midi_to_dataframe(fileorch)
dforch = dforch.sort_values(
    ['onset in quarter notes', 'duration in quarter notes', 'track number'],
    ascending=[True, True, True]
)
print(dforch.columns)
print("Table:\n", dforch)
np_orch = dforch.to_numpy()
mapping = learn_quaterna_mapping(np_orch, ytarget="track-channel")
print("Mapping:", mapping)

Orchestra file: midis/liszt_classical_archives-0/orchestra.mid
Index(['track number', 'track name', 'channel', 'program',
       'onset in quarter notes', 'duration in quarter notes', 'pitch',
       'velocity'],
      dtype='object')
Table:
       track number   track name  channel  program  onset in quarter notes  \
0                1        Flute        1       73                     0.0   
1                1        Flute        1       73                     0.0   
585              2         Oboe        2       68                     0.0   
586              2         Oboe        2       68                     0.0   
1206             3     Clarinet        3       71                     0.0   
...            ...          ...      ...      ...                     ...   
1205             2         Oboe        2       68                  1574.0   
4791            10       Violin       10       40                  1574.0   
5491            11        Viola       11       41               

In [4]:
print(f"Piano file: {filepiano}")
dfpiano = midi_to_dataframe(filepiano)
dfpiano = dfpiano.sort_values(
    ['onset in quarter notes', 'duration in quarter notes', 'track number'],
    ascending=[True, True, True]
)
print("Table:\n", dfpiano)
np_piano = dfpiano.to_numpy()
mapping = learn_quaterna_mapping(np_piano, ytarget="track-channel")
print("Mapping:", mapping)

Piano file: midis/liszt_classical_archives-0/piano.mid
Table:
       track number track name  channel  program  onset in quarter notes  \
0                0      Piano        0        0                     0.0   
1                0      Piano        0        0                     0.0   
2                0      Piano        0        0                     0.0   
3                0      Piano        0        0                     0.0   
4                0      Piano        0        0                     0.0   
...            ...        ...      ...      ...                     ...   
5290             0      Piano        0        0                  1572.0   
5291             0      Piano        0        0                  1572.0   
5292             0      Piano        0        0                  1572.0   
5293             0      Piano        0        0                  1572.0   
5294             0      Piano        0        0                  1572.0   

      duration in quarter notes  pit

In [5]:
import pandas as pd
import numpy as np
from collections import defaultdict

def map_notes_with_details(df1, df2, tol=0.05, transformations=None):
    """
    Map notes from df1 to df2 based on same pitch and onset,
    allowing small duration differences and multiple matches per note.

    Parameters
    ----------
    df1 : pd.DataFrame
        First table with columns:
        ['track number', 'track name', 'channel', 'program',
         'onset in quarter notes', 'duration in quarter notes', 'pitch', 'velocity']

    df2 : pd.DataFrame
        Second table with same columns as df1.

    tol : float
        Tolerance factor for duration matching (fractional, e.g. 0.05 = ±5%).

    Returns
    -------
    mapping : pd.DataFrame
        Table of matches with columns:
        ['df1_index', 'df2_index', 'onset_diff', 'duration_diff']

    stats : dict
        Dictionary with counts:
        {
            'matched_from_df1': int,
            'unmatched_from_df1': int,
            'matched_from_df2': int,
            'unmatched_from_df2': int
        }

    unmatched_df1 : pd.DataFrame
        Subset of df1 with notes that were not matched.

    matched_df2 : pd.DataFrame
        Subset of df2 with notes that were matched.

    unmatched_df2 : pd.DataFrame
        Subset of df2 with notes that were not matched.
    """

    # Build lookup that allows multiple df2 notes per (onset, pitch)
    df2_lookup = defaultdict(list)
    for idx, row in df2.iterrows():
        key = (row['onset in quarter notes'], row['pitch'])
        df2_lookup[key].append((idx, row['duration in quarter notes']))

    mappings = []
    mappings_transform = []
    matched_df1_indices = set()
    matched_df2_indices = set()
    matched_transform_df2_indices = set()

    for i, note1 in df1.iterrows():
        # ---- Direct matching ----
        key = (note1['onset in quarter notes'], note1['pitch'])
        direct_matched = False

        if key in df2_lookup:
            for df2_idx, df2_dur in df2_lookup[key]:
                dur_diff = abs(note1['duration in quarter notes'] - df2_dur)
                if dur_diff <= note1['duration in quarter notes'] * tol:
                    mappings.append({
                        'df1_index': i,
                        'df2_index': df2_idx,
                        'onset_diff': 0.0,
                        'duration_diff': dur_diff
                    })
                    matched_df2_indices.add(df2_idx)
                    direct_matched = True

        if direct_matched:
            matched_df1_indices.add(i)
            # continue  # Do not skip transformation if direct match found

        # ---- Transformation-based matching ----
        if transformations:
            for func, kwargs in transformations:
                transformed = func(note1, **kwargs)

                key_t = (transformed['onset in quarter notes'], transformed['pitch'])
                if key_t not in df2_lookup:
                    continue

                for df2_idx, df2_dur in df2_lookup[key_t]:
                    dur_diff = abs(transformed['duration in quarter notes'] - df2_dur)
                    if dur_diff <= transformed['duration in quarter notes'] * tol:
                        mappings_transform.append({
                            'df1_index': i,
                            'df2_index': df2_idx,
                            'onset_diff': abs(
                                transformed['onset in quarter notes']
                                - note1['onset in quarter notes']
                            ),
                            'duration_diff': dur_diff
                        })
                        matched_transform_df2_indices.add(df2_idx)
                        matched_df1_indices.add(i)

                        # Optionally break after first transformation match
                        # to avoid multiple transformed matches per note
                        # break


    mapping_df = pd.DataFrame(mappings)
    mapping_transform_df = pd.DataFrame(mappings_transform)


    unmatched_df1 = df1.loc[~df1.index.isin(matched_df1_indices)].copy()
    matched_df2 = df2.loc[df2.index.isin(matched_df2_indices)].copy()
    matched_transform_df2 = df2.loc[df2.index.isin(matched_transform_df2_indices)].copy()
    unmatched_df2 = df2.loc[
        ~df2.index.isin(matched_df2_indices.union(matched_transform_df2_indices))
    ].copy()

    stats = {
        'matched_from_df1': len(matched_df1_indices),
        'unmatched_from_df1': len(unmatched_df1),
        'matched_from_df2': len(matched_df2),
        'unmatched_from_df2': len(unmatched_df2),
        'matched_transform_from_df2': len(matched_transform_df2)
    }

    return (
        mapping_df,
        mapping_transform_df,
        stats,
        unmatched_df1,
        matched_df2,
        unmatched_df2,
        matched_transform_df2
    )



In [6]:
def transpose(note, n_semitones=12):
    # Example: transpose pitch
    new_note = note.copy()
    new_note['pitch'] = note['pitch'] + n_semitones
    return new_note

In [7]:
transformations = [
    (transpose, {'n_semitones': 12}),
    (transpose, {'n_semitones': 24}),
    (transpose, {'n_semitones': 46}),
    (transpose, {'n_semitones': -12}),
    (transpose, {'n_semitones': -24}),
    (transpose, {'n_semitones': -46}),
]


In [8]:
(
    mapping,
    mapping_transform,
    stats,
    df_only_piano,
    df_matched_orch,
    df_only_orch,
    df_matched_transform
) = map_notes_with_details(dfpiano, dforch, tol=1000, transformations=transformations)

print("Mapping statistics:")
for k, v in stats.items():
    print(f"{k}: {v}")

print("\nUnmatched from df1:", len(df_only_piano))
print("Matched from df2:", len(df_matched_orch))
print("Unmatched from df2:", len(df_only_orch))
print("Matched with transformation from df2:", len(df_matched_transform))


print(mapping)

Mapping statistics:
matched_from_df1: 3154
unmatched_from_df1: 2141
matched_from_df2: 3712
unmatched_from_df2: 1417
matched_transform_from_df2: 4198

Unmatched from df1: 2141
Matched from df2: 3712
Unmatched from df2: 1417
Matched with transformation from df2: 4198
      df1_index  df2_index  onset_diff  duration_diff
0             0       1675         0.0            0.0
1             0       6081         0.0            0.0
2             0       2369         0.0            1.0
3             1       1676         0.0            0.0
4             1       4792         0.0            0.0
...         ...        ...         ...            ...
3707       5245       2368         0.0           20.5
3708       5246       3774         0.0           20.5
3709       5254       4785         0.0            4.0
3710       5267       4787         0.0            2.0
3711       5283       4789         0.0            2.0

[3712 rows x 4 columns]


In [9]:
only_piano_filename = filepiano.replace("piano.mid", "only_piano.mid")
matched_orch_filename = fileorch.replace("orchestra.mid", "matched.mid")
only_orch_filename = fileorch.replace("orchestra.mid", "only_orchestra.mid")
matched_transform_orch_filename = fileorch.replace("orchestra.mid", "matched_transform.mid")


In [10]:
save_midi_with_exact_timing_structure(df_only_piano, only_piano_filename, reference_midi_path=filepiano)
save_midi_with_exact_timing_structure(df_matched_orch, matched_orch_filename, reference_midi_path=fileorch)
save_midi_with_exact_timing_structure(df_only_orch, only_orch_filename, reference_midi_path=fileorch)
save_midi_with_exact_timing_structure(df_matched_transform, matched_transform_orch_filename, reference_midi_path=fileorch)


=== SAVING WITH EXACT TIMING STRUCTURE ===
Extracting timing structure from midis/liszt_classical_archives-0/piano.mid
ticks_per_beat: 1024
Found 0 timing events:
Using ticks_per_beat: 1024
Applying key signature fix for MuseScore compatibility...
✓ Added consistent key signatures to 1 instrument tracks
✅ Saved midis/liszt_classical_archives-0/only_piano.mid with exact timing structure preserved
   - 0 timing events preserved
   - 1 instrument tracks created
   - Key signatures fixed for MuseScore compatibility

=== SAVING WITH EXACT TIMING STRUCTURE ===
Extracting timing structure from midis/liszt_classical_archives-0/orchestra.mid
ticks_per_beat: 1024
Found 0 timing events:
Using ticks_per_beat: 1024
Applying key signature fix for MuseScore compatibility...
✓ Added consistent key signatures to 10 instrument tracks
✅ Saved midis/liszt_classical_archives-0/matched.mid with exact timing structure preserved
   - 0 timing events preserved
   - 10 instrument tracks created
   - Key signat

In [11]:
#import matplotlib.pyplot as plt

def test_tolerance_sweep(df1, df2, transformations=None, tol_values=None):
    """
    Test the mapping function across a range of tolerance values and plot statistics.

    Parameters
    ----------
    df1 : pd.DataFrame
    df2 : pd.DataFrame
    transformations : list, optional
        List of (func, kwargs) for transformations.
    tol_values : list or np.ndarray, optional
        Sequence of tolerance values to test (default: np.logspace(-1, 2, 20))

    Returns
    -------
    results_df : pd.DataFrame
        A DataFrame containing stats for each tolerance value.
    """

    if tol_values is None:
        tol_values = np.logspace(-1, 2, 20)  # from 0.1 to 100

    results = []

    for tol in tol_values:
        (
            _,
            _,
            stats,
            _,
            _,
            _,
            _
        ) = map_notes_with_details(df1, df2, tol=tol, transformations=transformations)

        stats_row = {'tol': tol}
        stats_row.update(stats)
        results.append(stats_row)

    results_df = pd.DataFrame(results)

    # ---- Plotting ----
    #plt.figure(figsize=(10, 6))
    #for col in results_df.columns:
    #    if col != 'tol':
    #        plt.plot(results_df['tol'], results_df[col], label=col, marker='o')

    #plt.xscale('log')
    #plt.xlabel('Tolerance (tol)')
    #plt.ylabel('Count')
    #plt.title('Mapping Statistics vs Tolerance')
    #plt.legend()
    #plt.grid(True, which='both', linestyle='--', alpha=0.6)
    #plt.tight_layout()
    #plt.show()

    return results_df

# Run the tolerance test
results_df = test_tolerance_sweep(dfpiano, dforch, transformations=transformations)

# Inspect results
print(results_df.head())


        tol  matched_from_df1  unmatched_from_df1  matched_from_df2  \
0  0.100000              2072                3223              2412   
1  0.143845              2086                3209              2422   
2  0.206914              2106                3189              2441   
3  0.297635              2164                3131              2519   
4  0.428133              2208                3087              2557   

   unmatched_from_df2  matched_transform_from_df2  
0                3246                        2574  
1                3232                        2586  
2                3200                        2610  
3                3096                        2717  
4                3044                        2770  


In [12]:
print(results_df)

           tol  matched_from_df1  unmatched_from_df1  matched_from_df2  \
0     0.100000              2072                3223              2412   
1     0.143845              2086                3209              2422   
2     0.206914              2106                3189              2441   
3     0.297635              2164                3131              2519   
4     0.428133              2208                3087              2557   
5     0.615848              2457                2838              2810   
6     0.885867              2506                2789              2862   
7     1.274275              2762                2533              3172   
8     1.832981              2786                2509              3207   
9     2.636651              2915                2380              3337   
10    3.792690              3009                2286              3478   
11    5.455595              3055                2240              3556   
12    7.847600              3090      